# 30. SARSA
**Use case:** Supply Chain — Warehouse robot learning a safe route past a forklift lane

An e-commerce fulfilment centre runs mobile robots from a receiving dock to a packing station, and the shortest path runs right beside a forklift lane. Robots still explore and occasionally make random moves, and any step into the lane risks a collision, equipment damage and an OSHA-reportable incident. Data available: a grid map of aisles, racks and the forklift lane, move costs, and incident logs. Success means a learned route that keeps collisions near zero while robots keep exploring, with only a few extra moves per trip compared with the shortest path.

> 💡 A new forklift driver learns routes while sometimes taking a wrong turn. A careful driver (SARSA) remembers that 'I sometimes slip up, so I avoid the edge of the loading dock'. A bold driver (Q-learning) plans as if they never make mistakes and hugs the edge for the shortest route.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: One SARSA update by hand next to the forklift lane
Perform a single SARSA update on a tiny Q-table and compare it with the Q-learning update for the same experience.

| 5W + H | |
|---|---|
| **Who** | A supply-chain engineer meeting reinforcement learning for the first time. |
| **What** | A five-entry Q-table for two aisle cells, one experience where exploration picks a move into the forklift lane. |
| **When** | Before running a full grid: to see exactly why SARSA ends up more cautious than Q-learning. |
| **Where** | A plain Python script with a dict as the Q-table. |
| **Why** | The difference between SARSA and Q-learning is one line of maths; seeing it with numbers makes the safety argument concrete. |
| **How** | Compute the SARSA target R + gamma x Q(S',A') and the Q-learning target R + gamma x max Q(S',.), then apply alpha x TD error. |

**🧠 ML concepts:** State, action, reward, next state, next action, On-policy TD target, Off-policy (Q-learning) target, Learning rate alpha, Why exploration risk enters SARSA's values  
**🐍 Python concepts:** Dicts keyed by tuples, Tuple unpacking in for-loops over .items(), Multiple assignment on one line, Generator expressions with filters in max(), f-string width formatting

In [ ]:
# One SARSA update by hand: a warehouse robot next to the forklift lane (Easy)
# 🐍 PY: a dict keyed by (state, action) tuples is a perfectly good tiny Q-table
Q = {
    ("aisle_B2", "east"): -3.0,    # step along the edge of the forklift lane (short route)
    ("aisle_B2", "north"): -4.0,   # step up to the safe aisle (one extra move)
    ("aisle_B3", "east"): -2.0,    # keep going along the edge
    ("aisle_B3", "north"): -3.5,   # move away from the lane
    ("aisle_B3", "south"): -20.0,  # step INTO the forklift lane: collision risk

}
alpha, gamma, epsilon = 0.5, 1.0, 0.1
print("Q-table before:")
# 🐍 PY: .items() gives (key, value) pairs; the key itself is a tuple we unpack
for (s, a), v in Q.items():
    print(f"  Q({s}, {a:5s}) = {v:6.2f}")

# The robot is in aisle_B2, takes 'east', pays -1 for the move, lands in aisle_B3.
s, a, r, s_next = "aisle_B2", "east", -1.0, "aisle_B3"
# 🧠 ML: SARSA is ON-POLICY - it uses the action the robot will ACTUALLY take next (maybe exploratory)
# Suppose epsilon-greedy explored (10% chance) and picked 'south' - into the forklift lane
a_next = "south"
print(f"\nExperience: S={s}, A={a}, R={r}, S'={s_next}, A'={a_next}  <- the 5 letters S-A-R-S-A")

# 🧠 ML: SARSA target uses Q(S', A') of the action really chosen
sarsa_target = r + gamma * Q[(s_next, a_next)]
# 🧠 ML: Q-learning target would instead use the BEST next action, max over a' of Q(S', a')
# 🐍 PY: a generator expression filtered by an if, fed straight into max()
q_target = r + gamma * max(v for (st, _), v in Q.items() if st == s_next)
print(f"SARSA target      = {r} + {gamma} * Q({s_next}, {a_next}) = {sarsa_target:.2f}")
print(f"Q-learning target = {r} + {gamma} * max Q({s_next}, .)  = {q_target:.2f}")

# 🧠 ML: move the old estimate a fraction alpha toward the target
td_error = sarsa_target - Q[(s, a)]
new_q = Q[(s, a)] + alpha * td_error
print(f"\nTD error = {sarsa_target:.2f} - ({Q[(s, a)]:.2f}) = {td_error:.2f}")
print(f"Q({s}, east): {Q[(s, a)]:.2f} -> {new_q:.2f}   (Q-learning would give {Q[(s, a)] + alpha * (q_target - Q[(s, a)]):.2f})")
Q[(s, a)] = new_q
print("\nTakeaway: because SARSA 'remembers' that its own exploration sometimes goes wrong,")
print("it rates risky moves lower than Q-learning does - that is why it learns safer routes.")

**Try this next:**
- Change a_next to 'east' (no exploration this time) and compare both targets.
- Set alpha to 0.1: how far does Q(B2, east) move now?
- Add ('aisle_B3', 'west') with value -5 and check whether the Q-learning target changes.

## MEDIUM: Tabular SARSA on a 4x12 warehouse grid
Train tabular SARSA on a cliff-walking style warehouse grid and print the safe route the robot learns.

| 5W + H | |
|---|---|
| **Who** | A robotics engineer at a fulfilment centre prototyping route learning. |
| **What** | A 4x12 grid: dock bottom-left, packing station bottom-right, and a forklift lane along the bottom between them. |
| **When** | Prototype stage in simulation, before any physical robot follows a learned route. |
| **Where** | A numpy script with a 4x12x4 Q-table. |
| **Why** | The shortest route hugs the lane; with exploration still switched on, that route causes collisions. SARSA should learn to keep its distance. |
| **How** | 500 episodes of epsilon-greedy SARSA (alpha 0.5, epsilon 0.1), -1 per move, -100 and reset for entering the lane, then draw the greedy route. |

**🧠 ML concepts:** Episodic tasks, Tabular Q-values, Epsilon-greedy with random tie-breaking, SARSA update rule, Learning curves (reward and collisions per 100 episodes)  
**🐍 Python concepts:** Set comprehensions, Dicts mapping actions to moves, min/max clamping, Tuple indexing into numpy arrays, zip(*list) to unzip, Nested list comprehensions for a grid

In [ ]:
# Tabular SARSA: a warehouse robot learns a safe route past a forklift lane (Medium)
import numpy as np

np.random.seed(42)
ROWS, COLS = 4, 12
START, GOAL = (3, 0), (3, 11)          # bottom-left dock -> bottom-right packing station
# 🐍 PY: a set comprehension - every cell on the bottom row between start and goal is the forklift lane
HAZARD = {(3, c) for c in range(1, 11)}
MOVES = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}  # up, right, down, left
ARROWS = "^>v<"


def step(state, action):
    dr, dc = MOVES[action]
    # 🐍 PY: min/max clamp keeps the robot inside the warehouse walls
    r = min(max(state[0] + dr, 0), ROWS - 1)
    c = min(max(state[1] + dc, 0), COLS - 1)
    if (r, c) in HAZARD:
        # 🧠 ML: entering the forklift lane = near-miss/collision: big penalty, robot reset to the dock
        return START, -100, True
    return (r, c), -1, (r, c) == GOAL   # -1 per move encourages short routes


def choose(Q, s, eps):
    # 🧠 ML: epsilon-greedy - explore with probability eps, otherwise take the best known action
    if np.random.rand() < eps:
        return np.random.randint(4)
    # 🐍 PY: random tie-breaking among equal best actions avoids always picking 'up'
    best = np.flatnonzero(Q[s] == Q[s].max())
    return int(np.random.choice(best))


# 🧠 ML: Q-table = one row of 4 action-values per grid cell, shape (4, 12, 4)
Q = np.zeros((ROWS, COLS, 4))
alpha, gamma, eps = 0.5, 1.0, 0.1
history = []
for episode in range(500):
    s, total, collided = START, 0, False
    a = choose(Q, s, eps)
    for _ in range(200):
        s2, r, done = step(s, a)
        total += r
        collided |= (r == -100)
        a2 = choose(Q, s2, eps)
        # 🧠 ML: SARSA update uses the NEXT action a2 actually chosen by the same epsilon-greedy policy
        # 🐍 PY: conditional expression - a finished episode has no future value to add
        target = r if done else r + gamma * Q[s2][a2]
        Q[s][a] += alpha * (target - Q[s][a])
        if done:
            break
        # 🐍 PY: tuple assignment swaps in the new state and action in one line
        s, a = s2, a2
    history.append((total, collided))

# 🐍 PY: slicing the last 100 entries and unpacking with zip(*...)
rewards, cols = zip(*history[-100:])
print(f"Last 100 training episodes: avg reward {np.mean(rewards):.1f}, collisions {sum(cols)}")
for start in (0, 100, 200, 300, 400):
    chunk = history[start:start + 100]
    print(f"  episodes {start:3d}-{start + 99:3d}: avg reward {np.mean([h[0] for h in chunk]):7.1f}, "
          f"collisions {sum(h[1] for h in chunk):3d}")

# Follow the greedy policy and draw the route
grid = [["." for _ in range(COLS)] for _ in range(ROWS)]
for (r, c) in HAZARD:
    grid[r][c] = "F"
s, steps = START, 0
while s != GOAL and steps < 50:
    a = int(np.argmax(Q[s]))
    grid[s[0]][s[1]] = ARROWS[a]
    s, _, _ = step(s, a)
    steps += 1
grid[START[0]][START[1]], grid[GOAL[0]][GOAL[1]] = "S", "G"
print(f"\nGreedy route learned by SARSA ({steps} moves). F = forklift lane:")
print("\n".join(" ".join(row) for row in grid))
print("\nBusiness read: SARSA keeps a buffer row away from the forklift lane, accepting a few extra")
print("moves because the robot's own random exploration could otherwise drift into traffic.")

**Try this next:**
- Set eps to 0.01: does SARSA still choose the top-row route?
- Change the collision penalty to -10 and see whether the route moves closer to the lane.
- Replace Q[s2][a2] with Q[s2].max() (Q-learning) and compare the route.

## HARD: SARSA vs Q-learning vs Expected SARSA under exploration
Compare SARSA, Q-learning and Expected SARSA on a richer warehouse with racks, sweep the exploration rate, draw both routes and convert collisions into daily cost.

| 5W + H | |
|---|---|
| **Who** | A warehouse automation team deciding which learning rule to use in their robot simulator and presenting safety evidence to operations. |
| **What** | A 5x12 grid with storage racks, a forklift lane, optional wheel slip, three algorithms, four exploration levels and five seeds. |
| **When** | Before deployment, and whenever the layout changes or exploration settings are revisited. |
| **Where** | A numpy simulation; Q-tables are saved with np.savez for the fleet manager. |
| **Why** | The shortest route is not the safest when robots keep exploring. Operations needs collisions per trip and dollars, not only average reward. |
| **How** | Shared training and evaluation functions parameterised by method, multi-seed averages, an epsilon sweep, route maps and a cost model of $2,500 per incident. |

**🧠 ML concepts:** On-policy vs off-policy learning, Expected SARSA, Evaluating with the exploration that stays on in operation, Multi-seed averaging, Exploration-rate sweep as tuning, Turning RL metrics into business cost  
**🐍 Python concepts:** Classes wrapping an environment, Chained comparisons, Functions parameterised by a method string, Sets to detect loops, Dict comprehension with ** unpacking into np.savez, Thousands separators in f-strings

In [ ]:
# SARSA vs Q-learning (and Expected SARSA) for a warehouse robot near a forklift lane (Hard)
import os
import tempfile
import numpy as np

ROWS, COLS = 5, 12
START, GOAL = (4, 0), (4, 11)
HAZARD = {(4, c) for c in range(1, 11)}              # forklift lane along the bottom
RACKS = {(2, c) for c in range(3, 9) if c != 6}      # storage racks the robot cannot enter
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]
ARROWS = "^>v<"
COLLISION, MOVE_COST, SLIP = -100.0, -1.0, 0.0  # SLIP = chance of wheel slip (try 0.05)


class Warehouse:
    """Grid with racks, a forklift lane and optional wheel slip (robot moves randomly SLIP of the time)."""
    def __init__(self, rng):
        self.rng = rng

    def step(self, s, a):
        # 🧠 ML: with SLIP > 0 transitions become stochastic - the same action may give a different result
        if self.rng.random() < SLIP:
            a = int(self.rng.integers(4))
        r, c = s[0] + MOVES[a][0], s[1] + MOVES[a][1]
        # 🐍 PY: chained comparison 0 <= r < ROWS reads like maths
        if not (0 <= r < ROWS and 0 <= c < COLS) or (r, c) in RACKS:
            r, c = s                                   # bump into wall/rack: stay put
        if (r, c) in HAZARD:
            return START, COLLISION, True, True
        return (r, c), MOVE_COST, (r, c) == GOAL, False


def eps_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(4))
    best = np.flatnonzero(Q[s] == Q[s].max())
    return int(rng.choice(best))


def td_target(method, Q, r, s2, a2, done, gamma, eps):
    # 🧠 ML: the ONLY difference between the three algorithms is this one line of maths
    if done:
        return r
    if method == "SARSA":            # on-policy: value of the action we will really take
        return r + gamma * Q[s2][a2]
    if method == "Q-learning":       # off-policy: value of the best action, ignoring exploration
        return r + gamma * Q[s2].max()
    # Expected SARSA: average over the epsilon-greedy policy's action probabilities
    probs = np.full(4, eps / 4)
    probs[Q[s2].argmax()] += 1 - eps
    return r + gamma * probs @ Q[s2]


def train(method, episodes=500, alpha=0.5, gamma=1.0, eps=0.1, seed=0):
    rng = np.random.default_rng(seed)
    env, Q = Warehouse(rng), np.zeros((ROWS, COLS, 4))
    log = []
    for _ in range(episodes):
        s, total, hits = START, 0.0, 0
        a = eps_greedy(Q, s, eps, rng)
        for _ in range(300):
            s2, r, done, hit = env.step(s, a)
            a2 = eps_greedy(Q, s2, eps, rng)
            Q[s][a] += alpha * (td_target(method, Q, r, s2, a2, done, gamma, eps) - Q[s][a])
            total, hits = total + r, hits + hit
            if done:
                break
            s, a = s2, a2
        log.append((total, hits))
    return Q, np.array(log)


def evaluate(Q, eps, n=300, seed=123):
    # 🧠 ML: online evaluation - the robot still explores a little in production (eps>0) and still slips
    rng = np.random.default_rng(seed)
    env, rewards, hits, steps = Warehouse(rng), [], 0, []
    for _ in range(n):
        s, total = START, 0.0
        for k in range(300):
            s, r, done, hit = env.step(s, eps_greedy(Q, s, eps, rng))
            total, hits = total + r, hits + hit
            if done:
                break
        rewards.append(total)
        steps.append(k + 1)
    return np.mean(rewards), hits / n, np.mean(steps)


def draw_route(Q):
    grid = [["." for _ in range(COLS)] for _ in range(ROWS)]
    for r, c in HAZARD:
        grid[r][c] = "F"
    for r, c in RACKS:
        grid[r][c] = "#"
    s, seen = START, set()
    # 🐍 PY: a 'seen' set guards against an infinite loop if the greedy policy cycles
    while s != GOAL and s not in seen:
        seen.add(s)
        a = int(Q[s].argmax())
        grid[s[0]][s[1]] = ARROWS[a]
        r, c = s[0] + MOVES[a][0], s[1] + MOVES[a][1]
        if not (0 <= r < ROWS and 0 <= c < COLS) or (r, c) in RACKS:
            break
        s = (r, c)
    grid[START[0]][START[1]], grid[GOAL[0]][GOAL[1]] = "S", "G"
    return "\n".join("  " + " ".join(row) for row in grid), len(seen)


def main():
    methods = ["SARSA", "Q-learning", "Expected SARSA"]
    # 🧠 ML: average over 5 seeds - RL results from a single run are notoriously noisy
    print("Training (eps=0.1, alpha=0.5), mean over 5 seeds, last 100 episodes:")
    print(f"{'method':15s} {'train reward':>12s} {'collisions/ep':>14s}")
    tables = {}
    for m in methods:
        logs = [train(m, seed=sd) for sd in range(5)]
        tables[m] = logs[0][0]
        last = np.concatenate([lg[-100:] for _, lg in logs])
        print(f"{m:15s} {last[:, 0].mean():12.1f} {last[:, 1].mean():14.3f}")

    # 🧠 ML: tuning sweep - how does the exploration rate change the gap between SARSA and Q-learning?
    print("\nExploration sweep - evaluated WITH the same eps the robot keeps in operation:")
    print(f"{'eps':>5s} {'method':15s} {'avg reward':>11s} {'collisions/trip':>16s} {'avg moves':>10s}")
    for eps in (0.0, 0.05, 0.1, 0.2):
        for m in ("SARSA", "Q-learning"):
            Q, _ = train(m, eps=max(eps, 0.02), seed=1)
            rew, col, mv = evaluate(Q, eps)
            print(f"{eps:5.2f} {m:15s} {rew:11.1f} {col:16.3f} {mv:10.1f}")

    for m in ("SARSA", "Q-learning"):
        route, n = draw_route(tables[m])
        print(f"\n{m} greedy route ({n} moves). F=forklift lane, #=rack:")
        print(route)

    # 🧠 ML: business cost - convert collisions to dollars (equipment damage, stoppage, OSHA reporting)
    cost_per_hit, trips_per_day = 2500, 400
    print("\nDaily risk cost at eps=0.1 (400 trips/day, $2,500 per forklift incident):")
    for m in ("SARSA", "Q-learning"):
        _, col, mv = evaluate(tables[m], 0.1)
        print(f"  {m:11s}: {col * trips_per_day:5.1f} incidents/day -> ${col * trips_per_day * cost_per_hit:9,.0f}"
              f"  (avg {mv:.1f} moves/trip)")

    out = tempfile.mkdtemp()
    # 🐍 PY: np.savez stores several named arrays in one compressed file
    np.savez(os.path.join(out, "warehouse_q_tables.npz"), **{m.replace(" ", "_"): tables[m] for m in methods})
    print(f"\nSaved Q-tables to {out}")
    print("SAFETY NOTE: RL routing is a planning aid, not a safety system. Physical separation, speed limits,")
    print("lidar e-stops and ANSI/RIA R15.08 / OSHA 1910.178 controls must stay in place regardless of policy.")


if __name__ == "__main__":
    main()

**Try this next:**
- Set SLIP to 0.05 and rerun: how does wheel slip change the gap between the two methods?
- Remove the gap in the rack row (c != 6) and see how SARSA's route changes.
- Decay epsilon from 0.2 to 0.0 during training and check whether SARSA's route converges to Q-learning's.